# Delineate-Anything example

Run the `delineate_anything` openEO process on CDSE and download the resulting field boundaries as GeoParquet.

In [1]:
import openeo

eoconn = openeo.connect("https://openeo.dataspace.copernicus.eu").authenticate_oidc()

Authenticated using refresh token.


In [2]:
aoi = {
    "type": "Polygon",
    "coordinates": [[
        [4.997886465978867, 51.000003053347534],
        [5.0500127304842835, 50.99909971909893],
        [5.052258312621218, 51.04996441050298],
        [5.000075043015026, 51.05086937572316],
        [4.997886465978867, 51.000003053347534],
    ]],
}
date = ["2024-05-01", "2024-08-31"]

In [ ]:
fields = eoconn.datacube_from_process(
    "delineate_anything",
    namespace="https://raw.githubusercontent.com/Lavreniuk/Delineate-Anything/main/openeo_udp/process_graph/delineate_anything_udp.json",
    geometry=aoi,
    temporal_extent=date,
    iou_threshold=0.7,
    confidence_threshold=0.15,
)

job = fields.create_job(
    title="Delineate-Anything job example",
    job_options={
        "udf-dependency-archives": [
            "https://s3.waw3-1.cloudferro.com/project_dependencies/onnx_deps_python311.zip#onnx_deps",
            "https://s3.waw3-1.cloudferro.com/project_dependencies/DelineateAnythingv2.zip#onnx_models",
        ],
    },
    out_format="parquet",
)
job.start_and_wait()

0:00:00 Job 'j-2610271455424e399b308fa3ed226501': send 'start'
0:00:02 Job 'j-2610271455424e399b308fa3ed226501': queued (progress 0%)
0:00:07 Job 'j-2610271455424e399b308fa3ed226501': queued (progress 0%)
0:00:14 Job 'j-2610271455424e399b308fa3ed226501': queued (progress 0%)
0:00:22 Job 'j-2610271455424e399b308fa3ed226501': queued (progress 0%)
0:00:32 Job 'j-2610271455424e399b308fa3ed226501': running (progress N/A)
0:00:44 Job 'j-2610271455424e399b308fa3ed226501': running (progress N/A)
0:01:00 Job 'j-2610271455424e399b308fa3ed226501': running (progress N/A)
0:01:19 Job 'j-2610271455424e399b308fa3ed226501': running (progress N/A)
0:01:43 Job 'j-2610271455424e399b308fa3ed226501': running (progress N/A)


In [ ]:
job.get_results().download_files("delineate_anything_results")

In [ ]:
import geopandas as gpd
from pathlib import Path

parquet_file = next(Path("delineate_anything_results").glob("*.parquet"))
gdf = gpd.read_parquet(parquet_file)
print(len(gdf), "fields")
gdf.plot(edgecolor="black", figsize=(8, 8))